# MMP before/after counts -- General Response G2

Answers the three numbers Reviewer 1 (#10) and the Editor asked for:
1. Active/Inactive/Inconclusive counts, raw MMP assay vs. the new PubChem MMP summary assay (AID 720637).
2. How the dual-active count (active in both MMP and the viability counter-screen) changed.
3. Whether viability-only actives (cytotoxic but not MMP-active) now appear, as Reviewer 1 predicted.

**Before** = the raw MMP assay (AID 720635) + the raw viability counter-screen (AID 720634), the same
files already used for the original `qsar_mito` pipeline (`mito/qsar_mito.ipynb`, same `pd.read_excel` call).
**After** = the final, duplicate-averaged PubChem MMP summary assay (AID 720637) already used throughout
this Rebuttal (`Summary_1_models/cleaning/AID_720637_cleaned.csv`).

All joins use `PUBCHEM_CID`, matching the join key used everywhere else in this project (not InChIKey --
the salt-duplication issue found separately is a different, not-yet-fixed problem, out of scope here).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os
import pandas as pd

BASE = os.path.abspath('.')
PROJECT_ROOT = os.path.abspath(os.path.join(BASE, '..', '..'))  # qsar_prediction/
OUT_DIR = os.path.join(BASE, 'results')
os.makedirs(OUT_DIR, exist_ok=True)

RAW_MMP_PATH = os.path.join(PROJECT_ROOT, 'mito', 'AID_720635_cleaned.xlsx')
RAW_VIABILITY_PATH = os.path.join(PROJECT_ROOT, 'cleaned_datasets', 'AID_720634_cleaned.xlsx')
SUMMARY_PATH = os.path.join(BASE, 'cleaning', 'AID_720637_cleaned.csv')

COLS = ['PUBCHEM_CID', 'PUBCHEM_EXT_DATASOURCE_SMILES', 'PUBCHEM_ACTIVITY_SCORE', 'PUBCHEM_ACTIVITY_OUTCOME']

## 1. Active / Inactive / Inconclusive -- before vs. after

In [ ]:
raw_mmp = pd.read_excel(RAW_MMP_PATH, usecols=COLS)
summary = pd.read_csv(SUMMARY_PATH, usecols=COLS)

def outcome_counts(df, label):
    c = df['PUBCHEM_ACTIVITY_OUTCOME'].value_counts()
    return {'Dataset': label, 'Total': len(df),
            'Active': int(c.get('Active', 0)), 'Inactive': int(c.get('Inactive', 0)),
            'Inconclusive': int(c.get('Inconclusive', 0))}

before_after = pd.DataFrame([
    outcome_counts(raw_mmp, 'BEFORE -- raw MMP (AID 720635)'),
    outcome_counts(summary, 'AFTER -- MMP summary (AID 720637)'),
]).set_index('Dataset')
before_after['Active_pct'] = (100 * before_after['Active'] / before_after['Total']).round(2)
before_after

## 2. Dual actives (MMP active AND viability active) -- before vs. after

"Before" = raw MMP active compounds also active in the raw viability counter-screen.
"After" = of those same raw-MMP-active compounds, how many are STILL called Active once the
artifact-corrected summary assay (AID 720637) is used instead.

In [ ]:
raw_viab = pd.read_excel(RAW_VIABILITY_PATH, usecols=COLS)
n_viab_dup_cid = raw_viab['PUBCHEM_CID'].duplicated().sum()
print(f'Raw viability file: {len(raw_viab):,} rows, {raw_viab["PUBCHEM_CID"].nunique():,} unique CIDs '
      f'({n_viab_dup_cid} duplicate-CID rows dropped below, keeping the first occurrence)')
raw_viab = raw_viab.drop_duplicates('PUBCHEM_CID', keep='first')

mmp_r = raw_mmp.rename(columns={'PUBCHEM_ACTIVITY_OUTCOME': 'mmp_outcome', 'PUBCHEM_ACTIVITY_SCORE': 'mmp_score'})
viab_r = raw_viab.rename(columns={'PUBCHEM_ACTIVITY_OUTCOME': 'viability_outcome', 'PUBCHEM_ACTIVITY_SCORE': 'viability_score'})
merged = mmp_r.merge(viab_r[['PUBCHEM_CID', 'viability_outcome', 'viability_score']], on='PUBCHEM_CID', how='inner')
print(f'Compounds with both raw MMP and raw viability data: {len(merged):,}')

crosstab_before = pd.crosstab(merged['mmp_outcome'], merged['viability_outcome'], margins=True)
print('\nBEFORE -- raw MMP outcome x raw viability outcome:')
crosstab_before

In [ ]:
n_dual_before = int(((merged['mmp_outcome'] == 'Active') & (merged['viability_outcome'] == 'Active')).sum())
n_mmp_active_before = int((merged['mmp_outcome'] == 'Active').sum())
print(f'Dual actives BEFORE (raw MMP active AND raw viability active): {n_dual_before:,}')
print(f'  -- out of {n_mmp_active_before:,} raw-MMP-active compounds with matched viability data '
      f'({100*n_dual_before/n_mmp_active_before:.1f}%)')

# AFTER: the correct parallel measurement -- merge the CORRECTED summary outcome
# (all compounds, not just the ones that used to be raw-MMP-active) with the same
# raw viability outcome, and crosstab the same way as BEFORE.
summary_r = summary.rename(columns={'PUBCHEM_ACTIVITY_OUTCOME': 'summary_outcome'})[['PUBCHEM_CID', 'summary_outcome']]
merged_after = summary_r.merge(viab_r[['PUBCHEM_CID', 'viability_outcome']], on='PUBCHEM_CID', how='inner')
print(f'\nCompounds with both corrected-summary and raw viability data: {len(merged_after):,}')

crosstab_after = pd.crosstab(merged_after['summary_outcome'], merged_after['viability_outcome'], margins=True)
print('\nAFTER -- corrected summary outcome x raw viability outcome:')
print(crosstab_after)

n_dual_after = int(((merged_after['summary_outcome'] == 'Active') & (merged_after['viability_outcome'] == 'Active')).sum())
n_summary_active_matched = int((merged_after['summary_outcome'] == 'Active').sum())
print(f'\nDual actives AFTER (corrected-summary Active AND raw-viability Active): {n_dual_after:,}')
print(f'  -- out of {n_summary_active_matched:,} corrected-summary-active compounds with matched viability data '
      f'({100*n_dual_after/n_summary_active_matched:.1f}%)')
print(f'\nDual-active count change: {n_dual_before:,} -> {n_dual_after:,} ({n_dual_after - n_dual_before:+,})')
print(f'Dual-active RATE change (as % of active compounds): {100*n_dual_before/n_mmp_active_before:.1f}% -> {100*n_dual_after/n_summary_active_matched:.1f}%')

## 3. Viability-only actives -- do they now appear? (Reviewer 1's specific prediction)

Reviewer 1 (#10) predicted there would be few or no compounds that are cytotoxic (viability-active)
but NOT MMP-active -- i.e. that essentially all viability-active compounds also show up as MMP
"active" (an artifact of cytotoxicity, not true MMP disruption).

In [ ]:
# BEFORE: viability-active compounds whose raw MMP outcome was NOT Active
viab_active = merged[merged['viability_outcome'] == 'Active']
viab_only_before = viab_active[viab_active['mmp_outcome'] != 'Active']
viab_only_before_inactive_mmp = viab_active[viab_active['mmp_outcome'] == 'Inactive']

print(f'Viability-active compounds (matched to raw MMP data): {len(viab_active):,}')
print(f'  BEFORE -- viability-active but raw-MMP NOT Active (Inactive or Inconclusive): {len(viab_only_before):,}')
print(f'  BEFORE -- viability-active but raw-MMP Inactive specifically: {len(viab_only_before_inactive_mmp):,}')
print(viab_only_before['mmp_outcome'].value_counts())

# AFTER: same viability-active compounds, checked against the corrected summary outcome
viab_active_after = viab_active[['PUBCHEM_CID']].merge(summary_r, on='PUBCHEM_CID', how='left')
viab_only_after = viab_active_after[viab_active_after['summary_outcome'] != 'Active']
viab_only_after_inactive = viab_active_after[viab_active_after['summary_outcome'] == 'Inactive']

print(f'\n  AFTER -- viability-active but corrected-summary NOT Active: {len(viab_only_after):,}')
print(f'  AFTER -- viability-active but corrected-summary Inactive specifically: {len(viab_only_after_inactive):,}')
print(viab_only_after['summary_outcome'].value_counts(dropna=False))

## 4. Summary table for General Response G2, and save results

In [ ]:
summary_table = pd.DataFrame([
    {'Item': 'Active count', 'Before (raw MMP)': n_mmp_active_before if False else int(before_after.loc['BEFORE -- raw MMP (AID 720635)','Active']),
     'After (summary assay)': int(before_after.loc['AFTER -- MMP summary (AID 720637)','Active'])},
    {'Item': 'Dual actives (MMP+viability both Active)', 'Before (raw MMP)': n_dual_before, 'After (summary assay)': n_dual_after},
    {'Item': 'Viability-only actives (viability Active, MMP not Active)', 'Before (raw MMP)': len(viab_only_before), 'After (summary assay)': len(viab_only_after)},
]).set_index('Item')

print(summary_table.to_string())

before_after.to_csv(os.path.join(OUT_DIR, 'mmp_active_inactive_before_after.csv'))
summary_table.to_csv(os.path.join(OUT_DIR, 'mmp_g2_summary_table.csv'))
crosstab_before.to_csv(os.path.join(OUT_DIR, 'mmp_viability_crosstab_before.csv'))
print(f'\nSaved -> {OUT_DIR}/')